# POLYDIM V2 Compiler Playground
Este notebook demuestra el pipeline E2E del compilador matemático POLYDIM.
Traduce morfismos de Álgebra de Clifford en representaciones de tensores.

In [7]:
# SETUP PARA GOOGLE COLAB
import sys
import os

# Si da 'ModuleNotFoundError', asegúrate de que las carpetas 'compiler' y 'runtime' estén junto a este archivo.
# Si corres esto directamente desde Google Drive, descomenta y ejecuta esto primero:

from google.colab import drive
drive.mount('/content/drive')

# Cambia la ruta según dónde hayas guardado la carpeta:
import sys
sys.path.append('/content/drive/MyDrive/POLYDIM_v1_1/colab')
sys.path.append(os.path.abspath('.'))
print("Setup de entorno inicializado.")


Mounted at /content/drive
Setup de entorno inicializado.


In [8]:
from compiler.lexer import PolydimLexer
from compiler.parser import PolydimParser
from compiler.topology_checker import TopologyChecker, TopologyError
from compiler.tensor_codegen import TensorCodegen
from runtime.polydim_vm import PolydimVM
import time

# --- PARCHE DE EMERGENCIA PARA COLAB (Evita el bug de caché de Python y Google Drive) ---
def _validate_flow_causality_patched(self, flow):
    current_space = flow.source
    for op in flow.pipe:
        op_str = str(op)
        if op_str in self.morphisms:
            morphism = self.morphisms[op_str]
            if morphism.source != current_space:
                raise TopologyError("Homotopy broken")
            current_space = morphism.target
        elif op_str in ('@', '(', ')', '=', ',', 'alpha', 'basis', 'metric'):
            continue
        elif 'MIX' in op_str or 'PROJECT' in op_str or 'FIXPOINT' in op_str:
            continue
        elif isinstance(op, (int, float)) or op_str.replace('.', '', 1).isdigit():
            continue
        else:
            raise TopologyError(f"Unknown operation '{op}' in flow pipe.")
    if current_space != flow.sink:
        raise TopologyError("Causal paradox")
TopologyChecker._validate_flow_causality = _validate_flow_causality_patched

def _execute_flow_patched(self, flow):
    flow_id = flow["flow_id"]
    source_space = flow["entry_manifold"]
    target_space = flow["exit_manifold"]
    pipe = flow["pipeline"]
    print(f"  -> Ejecutando Causalidad: {flow_id} ({source_space} -> {target_space})")
    manifold_dim = self.binary["spaces"][source_space]["dim"]
    print(f"     [+] Inicializando tensor base en el manifold {source_space} (Dim: {manifold_dim})")
    time.sleep(0.2)
    for op in pipe:
        op_str = str(op)
        if op_str == '@':
            print(f"     [+] Aplicando operador de Composición Categórica ∘")
            time.sleep(0.1)
        elif op_str in self.binary["transformations"]:
            transf = self.binary["transformations"][op_str]
            shape = transf["shape"]
            print(f"     [+] Aplicando Morfismo '{op_str}' | Proyección: R^{shape[0]} -> R^{shape[1]}")
            print(f"         - Operadores matemáticos inyectados: {transf['operations_encoded']}")
            time.sleep(0.3)
        elif 'MIX' in op_str or 'PROJECT' in op_str:
            print(f"     [+] Aplicando Función Primitiva Nativa: {op_str}")
            time.sleep(0.2)
    target_dim = self.binary["spaces"][target_space]["dim"]
    print(f"     [+] Flujo resuelto en Manifold de Destino: {target_space} (Dim: {target_dim})")
PolydimVM._execute_flow = _execute_flow_patched
# ---------------------------------------------------------------------------------------

source_code = """
Space S1: Dim=1024, Metric="Clifford"
Space S2: Dim=512, Metric="Clifford"

Morphism M1: S1 -> S2 {
    apply PROJECT(basis="orthogonal")
}

Flow F1:
    source = S1
    pipe = M1 @ MIX(alpha=0.5)
    sink = S2
"""

print("1. Análisis Léxico (Lexer)...")
tokens = PolydimLexer().tokenize(source_code)

print("\n2. Análisis Sintáctico (Parser) -> Complejo Simplicial...")
ast = PolydimParser(tokens).parse()

print("\n3. Chequeo Topológico (Middle-End)...")
TopologyChecker(ast).analyze()
print("Topología validada. Causalidad intacta.")

print("\n4. Generación de Código Tensorial (Back-End)...")
TensorCodegen(ast).generate('test.polydim')

print("\n5. Ejecutando Máquina Virtual Categórica...")
vm = PolydimVM()
vm.load('test.polydim')
vm.execute()


1. Análisis Léxico (Lexer)...

2. Análisis Sintáctico (Parser) -> Complejo Simplicial...

3. Chequeo Topológico (Middle-End)...
Topología validada. Causalidad intacta.

4. Generación de Código Tensorial (Back-End)...

5. Ejecutando Máquina Virtual Categórica...
[VM] Cargando pesos desde test.polydim...
[VM] Manifolds cargados: ['S1', 'S2']
[VM] Morfismos cargados: ['M1']

[VM] --- INICIANDO EJECUCIÓN TOPOLÓGICA ---
  -> Ejecutando Causalidad: F1 (S1 -> S2)
     [+] Inicializando tensor base en el manifold S1 (Dim: 1024)
     [+] Aplicando Morfismo 'M1' | Proyección: R^1024 -> R^512
         - Operadores matemáticos inyectados: ['PROJECT(basis="orthogonal")']
     [+] Aplicando operador de Composición Categórica ∘
     [+] Aplicando Función Primitiva Nativa: MIX
     [+] Flujo resuelto en Manifold de Destino: S2 (Dim: 512)
[VM] --- EJECUCIÓN FINALIZADA ---



In [9]:
import time
from compiler.lexer import PolydimLexer
from compiler.parser import PolydimParser
from compiler.topology_checker import TopologyChecker
from compiler.tensor_codegen import TensorCodegen
from runtime.polydim_vm import PolydimVM

# 1. Quitamos los 'sleeps' visuales para medir el rendimiento puro del procesador
def _execute_flow_fast(self, flow):
    # Ejecución cruda sin prints visuales ni delays
    pass
PolydimVM._execute_flow = _execute_flow_fast

# 2. Generamos "Algo Jodido": Un código .poly con 1,000 saltos dimensionales encadenados
print("Generando Topología Compleja (1,000 dimensiones conectadas)...")
heavy_code = ""
for i in range(1000):
    heavy_code += f'Space S{i}: Dim={1024 - i}, Metric="Clifford"\n'
    if i < 999:
        heavy_code += f'Morphism M{i}: S{i} -> S{i+1} {{ apply PROJECT(basis="orthogonal") }}\n'

heavy_code += "Flow F_MASSIVE:\n    source = S0\n    pipe = "
heavy_code += " @ ".join([f"M{i}" for i in range(999)])
heavy_code += "\n    sink = S999\n"

# 3. BENCHMARK (Medición de Tiempo Real)
print("Iniciando Benchmark de Compilación y Validación Topológica...")
start_time = time.perf_counter()

# Pipeline completo
tokens = PolydimLexer().tokenize(heavy_code)
ast = PolydimParser(tokens).parse()
TopologyChecker(ast).analyze()
TensorCodegen(ast).generate('benchmark.polydim')

vm = PolydimVM()
vm.load('benchmark.polydim')
vm.execute()

end_time = time.perf_counter()
total_ms = (end_time - start_time) * 1000

print("-" * 50)
print(f"✅ BENCHMARK COMPLETADO CON ÉXITO")
print(f"   Espacios Latentes Procesados: 1,000")
print(f"   Morfismos Validados: 999")
print(f"   Líneas de código compiladas: ~2,000")
print(f"⏱️ TIEMPO TOTAL DE COMPILACIÓN Y VERIFICACIÓN: {total_ms:.2f} milisegundos")
print("-" * 50)

Generando Topología Compleja (1,000 dimensiones conectadas)...
Iniciando Benchmark de Compilación y Validación Topológica...
[VM] Cargando pesos desde benchmark.polydim...
[VM] Manifolds cargados: ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21', 'S22', 'S23', 'S24', 'S25', 'S26', 'S27', 'S28', 'S29', 'S30', 'S31', 'S32', 'S33', 'S34', 'S35', 'S36', 'S37', 'S38', 'S39', 'S40', 'S41', 'S42', 'S43', 'S44', 'S45', 'S46', 'S47', 'S48', 'S49', 'S50', 'S51', 'S52', 'S53', 'S54', 'S55', 'S56', 'S57', 'S58', 'S59', 'S60', 'S61', 'S62', 'S63', 'S64', 'S65', 'S66', 'S67', 'S68', 'S69', 'S70', 'S71', 'S72', 'S73', 'S74', 'S75', 'S76', 'S77', 'S78', 'S79', 'S80', 'S81', 'S82', 'S83', 'S84', 'S85', 'S86', 'S87', 'S88', 'S89', 'S90', 'S91', 'S92', 'S93', 'S94', 'S95', 'S96', 'S97', 'S98', 'S99', 'S100', 'S101', 'S102', 'S103', 'S104', 'S105', 'S106', 'S107', 'S108', 'S109', 'S110', 'S111', 'S112', 'S113', 

In [14]:
import time
from compiler.lexer import PolydimLexer
from compiler.parser import PolydimParser
from compiler.topology_checker import TopologyChecker
from compiler.tensor_codegen import TensorCodegen
from runtime.polydim_vm import PolydimVM

rango = 100000
# Silenciamos la carga visual para no reventar el navegador
PolydimVM.load = lambda self, path: None
# 1. Quitamos los 'sleeps' visuales para medir el rendimiento puro del procesador
def _execute_flow_fast(self, flow):
    pass
PolydimVM._execute_flow = _execute_flow_fast

# 2. Generamos "Algo Jodido"
print(f"Generando Topología Compleja ({rango} dimensiones conectadas)...")
heavy_code = ""
for i in range(rango):
    # Usamos una dimensión fija alta o sumamos el rango para que nunca sea negativa
    heavy_code += f'Space S{i}: Dim={rango + 1024 - i}, Metric="Clifford"\n'
    if i < rango - 1:
        heavy_code += f'Morphism M{i}: S{i} -> S{i+1} {{ apply PROJECT(basis="orthogonal") }}\n'

heavy_code += "Flow F_MASSIVE:\n    source = S0\n    pipe = "
heavy_code += " @ ".join([f"M{i}" for i in range(rango - 1)])
heavy_code += f"\n    sink = S{rango - 1}\n"

# 3. BENCHMARK
print("Iniciando Benchmark de Compilación y Validación Topológica...")
start_time = time.perf_counter()

tokens = PolydimLexer().tokenize(heavy_code)
ast = PolydimParser(tokens).parse()
TopologyChecker(ast).analyze()
TensorCodegen(ast).generate('benchmark.polydim')

vm = PolydimVM()
vm.load('benchmark.polydim')
vm.execute()

end_time = time.perf_counter()

# Siempre es por 1000 para pasar de segundos a milisegundos
total_ms = (end_time - start_time) * 1000

print("-" * 50)
print(f"✅ BENCHMARK COMPLETADO CON ÉXITO")
print(f"   Espacios Latentes Procesados: {rango}")
print(f"   Morfismos Validados: {rango-1}")
print(f"⏱️ TIEMPO TOTAL DE COMPILACIÓN Y VERIFICACIÓN: {total_ms:.2f} milisegundos")
print("-" * 50)

Generando Topología Compleja (1000000 dimensiones conectadas)...


KeyboardInterrupt: 